# Shared classification workflow

Edit these cells in AWS Jupyter. Notebooks 06–08 load the saved definitions into their own kernel. Running this notebook defines the workflow without starting training.

In [ ]:
from pathlib import Path
import os

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "Notebooks/01_dataset_inventory.ipynb").is_file())
os.chdir(root)

## Image loading and whole-image preprocessing

In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image, ImageOps
from torch.utils.data import Dataset
from torchvision import transforms

ROOT = root
MANIFEST = Path(
    "reports/eda/exp16_duplicate_detection/dataset_split_manifest_corrected.csv"
)
VARIANTS = ("original_rgb", "lab_clahe", "bilateral_clahe")
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def load_manifest():
    path = ROOT / MANIFEST
    if not path.exists():
        path = ROOT / "Notebooks" / MANIFEST
    df = pd.read_csv(path)
    df["class_id"] = df.folder_2.str.extract(r"^(\d+)")[0].astype(int)
    if (
        not df.sample_id.is_unique
        or not df.s3_key.is_unique
        or set(df.split) != {"train", "validation", "test"}
        or set(df.class_id) != set(range(11))
    ):
        raise ValueError(
            "Invalid sample identifiers, splits or classes in corrected manifest"
        )
    return df


def evaluation_transform(size=224, mean=MEAN, std=STD):
    return transforms.Compose(
        [
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    )


class SeamImages(Dataset):
    def __init__(self, frame, variant, training=False):
        if variant not in VARIANTS:
            raise ValueError(variant)
        self.frame = frame.reset_index(drop=True)
        suffixes = [Path(key).relative_to("data") for key in frame.s3_key]
        roots = (
            [
                ROOT / "data",
                ROOT / "data/training_cache/original/data",
                ROOT / "Notebooks/data/training_cache/original_rgb",
            ]
            if variant == "original_rgb"
            else [ROOT / "data/training_cache/enhanced" / variant]
        )
        self.paths = []
        for suffix in suffixes:
            path = next(
                (base / suffix for base in roots if (base / suffix).is_file()), None
            )
            if path is None:
                raise FileNotFoundError(roots[0] / suffix)
            self.paths.append(path)
        steps = evaluation_transform().transforms
        if training:
            steps[1:1] = [
                transforms.RandomHorizontalFlip(),
                transforms.RandomAffine(
                    degrees=4, translate=(0.02, 0.02), scale=(0.97, 1.03), fill=128
                ),
                transforms.RandomApply(
                    [transforms.ColorJitter(0.08, 0.08, 0.05, 0.01)], p=0.5
                ),
            ]
        self.transform = transforms.Compose(steps)
        self.labels = self.frame.class_id.tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            return (
                self.transform(ImageOps.exif_transpose(image).convert("RGB")),
                self.labels[index],
            )

## Restore missing S3 images

When a cache is missing, call `restore_images("original_rgb")`, `restore_images("lab_clahe")`, or `restore_images("bilateral_clahe")` in this notebook after running the definition.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from uuid import uuid4

def restore_images(variant, bucket="amal-stitchingnet-data-903473565738-ap-southeast-2-an"):
    import boto3
    if variant not in VARIANTS:
        raise ValueError(variant)
    s3 = boto3.client("s3", region_name="ap-southeast-2")

    def download(key):
        relative = Path(key).relative_to("data")
        if variant == "original_rgb":
            destination = ROOT / key
            if any(
                (base / relative).is_file()
                for base in (
                    ROOT / "data/training_cache/original/data",
                    ROOT / "Notebooks/data/training_cache/original_rgb",
                )
            ):
                return
        else:
            destination = (
                ROOT / "data/training_cache/enhanced" / variant / relative
            )
            key = f"processed-data/exp17_multiview/{variant}/{relative.as_posix()}"
        if destination.is_file() and destination.stat().st_size:
            return
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_suffix("." + uuid4().hex + ".part")
        try:
            s3.download_file(bucket, key, str(temporary))
            temporary.replace(destination)
        finally:
            temporary.unlink(missing_ok=True)

    with ThreadPoolExecutor(max_workers=8) as pool:
        list(pool.map(download, load_manifest().s3_key))

In [ ]:
import json
import random
from datetime import datetime, timezone
from uuid import uuid4

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from torch import nn
from torch.utils.data import DataLoader



from torch.nn import functional as F
from torchvision import models

## Experiment settings

In [ ]:
CONFIGS = {
  "densenet121_fpn": {
    "batch_size": 16,
    "accumulation_steps": 1,
    "max_epochs": 60,
    "early_stopping_patience": 10,
    "backbone_lr": 3e-05,
    "head_lr": 0.0003,
    "weight_decay": 0.0001,
    "dropout": 0.4,
    "label_smoothing": 0.03,
    "scheduler_patience": 3,
    "scheduler_factor": 0.5,
    "min_backbone_lr": 1e-06,
    "min_head_lr": 1e-05
  },
  "convnext_tiny_fpn": {
    "batch_size": 8,
    "accumulation_steps": 2,
    "max_epochs": 70,
    "early_stopping_patience": 12,
    "backbone_lr": 1e-05,
    "head_lr": 0.0002,
    "weight_decay": 0.05,
    "dropout": 0.3,
    "label_smoothing": 0.08,
    "scheduler_patience": 4,
    "scheduler_factor": 0.5,
    "min_backbone_lr": 5e-07,
    "min_head_lr": 5e-06
  },
  "efficientnet_b0_fpn": {
    "batch_size": 24,
    "accumulation_steps": 1,
    "max_epochs": 60,
    "early_stopping_patience": 10,
    "backbone_lr": 8e-05,
    "head_lr": 0.0004,
    "weight_decay": 1e-05,
    "dropout": 0.25,
    "label_smoothing": 0.05,
    "scheduler_patience": 3,
    "scheduler_factor": 0.5,
    "min_backbone_lr": 1e-06,
    "min_head_lr": 1e-05
  },
  "resnet18_fpn": {
    "batch_size": 64,
    "accumulation_steps": 1,
    "max_epochs": 60,
    "min_epochs": 10,
    "early_stopping_patience": 10,
    "backbone_lr": 5e-05,
    "head_lr": 0.0003,
    "weight_decay": 0.0001,
    "dropout": 0.3,
    "label_smoothing": 0.03,
    "scheduler_patience": 3,
    "scheduler_factor": 0.5,
    "min_backbone_lr": 1e-06,
    "min_head_lr": 1e-05
  }
}

BACKBONES = (
    "resnet18_fpn",
    "densenet121_fpn",
    "convnext_tiny_fpn",
    "efficientnet_b0_fpn",
)

## Shared feature pyramid head

In [ ]:
class GeM(nn.Module):

    def __init__(self, p=3.0, eps=1e-06):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        p = torch.clamp(self.p, min=1.0, max=6.0)
        x = x.clamp(min=self.eps).pow(p)
        x = F.adaptive_avg_pool2d(x, 1)
        return x.pow(1.0 / p).flatten(1)


class FPNHead(nn.Module):

    def __init__(self, input_channels, num_classes=11, fpn_channels=256, dropout=0.3):
        super().__init__()

        def lateral_block(in_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, fpn_channels, kernel_size=1, bias=False),
                nn.GroupNorm(32, fpn_channels),
                nn.ReLU(inplace=True),
            )

        def smooth_block():
            return nn.Sequential(
                nn.Conv2d(
                    fpn_channels, fpn_channels, kernel_size=3, padding=1, bias=False
                ),
                nn.GroupNorm(32, fpn_channels),
                nn.ReLU(inplace=True),
            )

        self.lateral = nn.ModuleList([lateral_block(c) for c in input_channels])
        self.smooth = nn.ModuleList([smooth_block() for _ in range(4)])
        self.dropout2d = nn.Dropout2d(0.1)
        self.gem = GeM()
        self.classifier = nn.Sequential(
            nn.Linear(fpn_channels * 4, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, c2, c3, c4, c5):
        features = [c2, c3, c4, c5]
        l2, l3, l4, l5 = [self.lateral[i](features[i]) for i in range(4)]
        p5 = l5
        p4 = l4 + F.interpolate(p5, size=l4.shape[-2:], mode="nearest")
        p3 = l3 + F.interpolate(p4, size=l3.shape[-2:], mode="nearest")
        p2 = l2 + F.interpolate(p3, size=l2.shape[-2:], mode="nearest")
        pyramid = [
            self.smooth[0](p2),
            self.smooth[1](p3),
            self.smooth[2](p4),
            self.smooth[3](p5),
        ]
        pooled = [self.gem(self.dropout2d(feature)) for feature in pyramid]
        combined = torch.cat(pooled, dim=1)
        return self.classifier(combined)

## DenseNet, ConvNeXt and EfficientNet backbones

In [ ]:
class FPNClassifier(nn.Module):

    def __init__(self, architecture, num_classes=11, pretrained=True, dropout=0.3):
        super().__init__()
        self.architecture = architecture
        if architecture == "densenet121_fpn":
            weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
            backbone = models.densenet121(weights=weights)
            backbone.classifier = nn.Identity()
            channels = [256, 512, 1024, 1024]
            self.backbone_type = "densenet121"
        elif architecture == "convnext_tiny_fpn":
            weights = models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
            backbone = models.convnext_tiny(weights=weights)
            backbone.classifier = nn.Identity()
            channels = [96, 192, 384, 768]
            self.backbone_type = "convnext_tiny"
        elif architecture == "efficientnet_b0_fpn":
            weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
            backbone = models.efficientnet_b0(weights=weights)
            backbone.classifier = nn.Identity()
            channels = [24, 40, 112, 1280]
            self.backbone_type = "efficientnet_b0"
        else:
            raise ValueError(f"Unknown architecture: {architecture}")
        self.backbone = backbone
        self.fpn_head = FPNHead(
            input_channels=channels,
            num_classes=num_classes,
            fpn_channels=256,
            dropout=dropout,
        )

    def extract_features(self, x):
        f = self.backbone.features
        if self.backbone_type == "densenet121":
            x = f.conv0(x)
            x = f.norm0(x)
            x = f.relu0(x)
            x = f.pool0(x)
            c2 = f.denseblock1(x)
            x = f.transition1(c2)
            c3 = f.denseblock2(x)
            x = f.transition2(c3)
            c4 = f.denseblock3(x)
            x = f.transition3(c4)
            c5 = f.denseblock4(x)
            c5 = F.relu(f.norm5(c5), inplace=False)
        elif self.backbone_type == "convnext_tiny":
            x = f[0](x)
            c2 = f[1](x)
            x = f[2](c2)
            c3 = f[3](x)
            x = f[4](c3)
            c4 = f[5](x)
            x = f[6](c4)
            c5 = f[7](x)
        elif self.backbone_type == "efficientnet_b0":
            x = f[0](x)
            x = f[1](x)
            c2 = f[2](x)
            c3 = f[3](c2)
            x = f[4](c3)
            c4 = f[5](x)
            x = f[6](c4)
            x = f[7](x)
            c5 = f[8](x)
        return (c2, c3, c4, c5)

    def forward(self, x):
        features = self.extract_features(x)
        return self.fpn_head(*features)

## ResNet18 feature pyramid

In [ ]:
class ResNet18FPN(nn.Module):

    def __init__(self, num_classes=11, fpn_channels=128, dropout=0.3, pretrained=True):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet18(weights=weights)
        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.lateral2 = nn.Conv2d(64, fpn_channels, kernel_size=1)
        self.lateral3 = nn.Conv2d(128, fpn_channels, kernel_size=1)
        self.lateral4 = nn.Conv2d(256, fpn_channels, kernel_size=1)
        self.lateral5 = nn.Conv2d(512, fpn_channels, kernel_size=1)
        self.smooth2 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)
        self.smooth3 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)
        self.smooth4 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)
        self.smooth5 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)
        self.classifier = nn.Sequential(
            nn.Linear(fpn_channels * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        c2 = self.layer1(x)
        c3 = self.layer2(c2)
        c4 = self.layer3(c3)
        c5 = self.layer4(c4)
        p5 = self.lateral5(c5)
        p4 = self.lateral4(c4)
        p3 = self.lateral3(c3)
        p2 = self.lateral2(c2)
        p4 = p4 + nn.functional.interpolate(p5, size=p4.shape[-2:], mode="nearest")
        p3 = p3 + nn.functional.interpolate(p4, size=p3.shape[-2:], mode="nearest")
        p2 = p2 + nn.functional.interpolate(p3, size=p2.shape[-2:], mode="nearest")
        p5 = self.smooth5(p5)
        p4 = self.smooth4(p4)
        p3 = self.smooth3(p3)
        p2 = self.smooth2(p2)
        f2 = nn.functional.adaptive_avg_pool2d(p2, output_size=1).flatten(1)
        f3 = nn.functional.adaptive_avg_pool2d(p3, output_size=1).flatten(1)
        f4 = nn.functional.adaptive_avg_pool2d(p4, output_size=1).flatten(1)
        f5 = nn.functional.adaptive_avg_pool2d(p5, output_size=1).flatten(1)
        fused = torch.cat([f2, f3, f4, f5], dim=1)
        output = self.classifier(fused)
        return output

## Build the classifier

In [ ]:
def build_model(name, dropout, pretrained=True):
    if name == "resnet18_baseline":
        model = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT if pretrained else None
        )
        model.fc = nn.Linear(model.fc.in_features, 11)
        return model
    if name == "resnet18_fpn":
        return ResNet18FPN(dropout=dropout, pretrained=pretrained)
    return FPNClassifier(name, num_classes=11, pretrained=pretrained, dropout=dropout)

## Evaluate predictions and export error reports

In [ ]:
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss


@torch.inference_mode()
def evaluate(model, loader, device, folder):
    """Collect predictions once and derive every report from those rows."""
    model.eval()
    batches = []
    inference_seconds = 0.0
    for images, _ in loader:
        start = perf_counter()
        batches.append(model(images.to(device)).float().softmax(1).cpu())
        inference_seconds += perf_counter() - start
    probabilities = torch.cat(batches).numpy()
    rows = (
        loader.dataset.frame[
            ["sample_id", "s3_key", "folder_1", "folder_2", "class_id"]
        ]
        .rename(columns={"folder_1": "fabric", "folder_2": "actual_class"})
        .copy()
    )
    rows["predicted_class_id"] = probabilities.argmax(1)
    rows["predicted_class"] = rows.predicted_class_id.map(
        dict(zip(rows.class_id, rows.actual_class))
    )
    rows["confidence"] = probabilities.max(1)
    rows["correct"] = rows.class_id == rows.predicted_class_id
    folder.mkdir(parents=True, exist_ok=True)
    for i in range(probabilities.shape[1]):
        rows[f"prob_class_{i}"] = probabilities[:, i]
    rows.to_csv(folder / "all_predictions.csv", index=False)
    rows.loc[~rows.correct].sort_values("confidence", ascending=False).to_csv(
        folder / "wrong_predictions.csv", index=False
    )
    metrics = dict(
        images=len(rows),
        device=str(device),
        batch_size=loader.batch_size,
        inference_images_per_second=len(rows) / inference_seconds,
        wrong=int((~rows.correct).sum()),
        defects_called_normal=int(
            ((rows.class_id != 0) & (rows.predicted_class_id == 0)).sum()
        ),
        accuracy=accuracy_score(rows.class_id, rows.predicted_class_id),
        balanced_accuracy=balanced_accuracy_score(
            rows.class_id, rows.predicted_class_id
        ),
        macro_f1=f1_score(rows.class_id, rows.predicted_class_id, average="macro"),
        log_loss=log_loss(
            rows.class_id, probabilities, labels=np.arange(probabilities.shape[1])
        ),
    )
    for group in ["actual_class", "fabric"]:
        summaries = []
        for name, part in rows.groupby(group):
            summaries.append(
                {
                    group: name,
                    "images": len(part),
                    "wrong": int((~part.correct).sum()),
                    (
                        "recall" if group == "actual_class" else "accuracy"
                    ): part.correct.mean(),
                }
            )
        pd.DataFrame(summaries).to_csv(
            folder / f"performance_by_{group}.csv", index=False
        )
    pd.DataFrame([metrics]).to_csv(folder / "metrics.csv", index=False)
    return metrics

## Train and save the best validation checkpoint

In [ ]:
def train(name, variant="original_rgb"):
    is_resnet = name in ("resnet18_baseline", "resnet18_fpn")
    cfg = CONFIGS["resnet18_fpn" if is_resnet else name]
    seed = 42
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    frame = load_manifest()
    loaders = {}
    for split in ("train", "validation", "test"):
        dataset = SeamImages(
            frame.loc[frame.split == split], variant, training=split == "train"
        )
        loaders[split] = DataLoader(
            dataset, batch_size=cfg["batch_size"], shuffle=split == "train"
        )
    model = build_model(name, cfg["dropout"]).to(device)
    head_prefixes = ("fc.", "classifier.", "lateral", "smooth", "fpn_head.")
    backbone, head = [], []
    for key, parameter in model.named_parameters():
        (head if key.startswith(head_prefixes) else backbone).append(parameter)
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone, "lr": cfg["backbone_lr"]},
            {"params": head, "lr": cfg["head_lr"]},
        ],
        weight_decay=cfg["weight_decay"],
    )
    weights = None
    if is_resnet:
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=[cfg["backbone_lr"], cfg["head_lr"]],
            epochs=cfg["max_epochs"],
            steps_per_epoch=len(loaders["train"]),
        )
    else:
        counts = torch.bincount(
            torch.tensor(loaders["train"].dataset.labels), minlength=11
        ).float()
        weights = counts.rsqrt()
        weights = (weights / weights.mean()).to(device)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            patience=cfg["scheduler_patience"],
            factor=cfg["scheduler_factor"],
            min_lr=[cfg["min_backbone_lr"], cfg["min_head_lr"]],
        )
    criterion = nn.CrossEntropyLoss(
        weight=weights, label_smoothing=cfg["label_smoothing"]
    )
    run_id = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
    )
    folder = ROOT / "models" / "runs" / name / variant / run_id
    folder.mkdir(parents=True)
    metadata = dict(
        architecture=name,
        input_variant=variant,
        config=cfg,
        seed=seed,
        scheduler=type(scheduler).__name__,
        image_size=224,
        imagenet_mean=MEAN,
        imagenet_std=STD,
        num_classes=11,
        class_names=frame.sort_values("class_id").folder_2.drop_duplicates().tolist(),
    )
    (folder / "config.json").write_text(json.dumps(metadata, indent=2))
    frame[["sample_id", "s3_key", "split", "class_id"]].to_csv(
        folder / "split_manifest.csv", index=False
    )
    best, best_epoch = -1, 0
    history = []
    for epoch in range(1, cfg["max_epochs"] + 1):
        model.train()
        optimizer.zero_grad()
        batches = len(loaders["train"])
        accumulation = cfg["accumulation_steps"]
        for step, (images, labels) in enumerate(loaders["train"]):
            logits = model(images.to(device))
            group_size = min(
                accumulation, batches - (step // accumulation) * accumulation
            )
            (criterion(logits, labels.to(device)) / group_size).backward()
            if (step + 1) % accumulation == 0 or step + 1 == batches:
                optimizer.step()
                optimizer.zero_grad()
                if is_resnet:
                    scheduler.step()
        model.eval()
        with torch.inference_mode():
            predictions = torch.cat(
                [
                    model(images.to(device)).argmax(1).cpu()
                    for images, _ in loaders["validation"]
                ]
            )
        score = f1_score(
            loaders["validation"].dataset.labels, predictions, average="macro"
        )
        if not is_resnet:
            scheduler.step(score)
        history.append({"epoch": epoch, "validation_macro_f1": score})
        pd.DataFrame(history).to_csv(folder / "training_history.csv", index=False)
        if score > best:
            best, best_epoch = score, epoch
            torch.save(
                dict(
                    **metadata,
                    model_state_dict=model.state_dict(),
                    epoch=epoch,
                    validation_macro_f1=float(score),
                ),
                folder / "best_model.pth",
            )
        if (
            epoch >= cfg.get("min_epochs", 1)
            and epoch - best_epoch >= cfg["early_stopping_patience"]
        ):
            break
    checkpoint = torch.load(
        folder / "best_model.pth", map_location=device, weights_only=True
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    for split in ("validation", "test"):
        evaluate(model, loaders[split], device, folder / "evaluation" / split)
    return folder

## Compare the completed runs

In [ ]:
def leading_rgb_backbone():
    scores = {}
    for name in BACKBONES:
        paths = sorted(
            (ROOT / "models" / "runs" / name / "original_rgb").glob(
                "*/evaluation/validation/metrics.csv"
            )
        )
        if not paths:
            raise FileNotFoundError(f"Complete the original RGB run for {name} first.")
        scores[name] = pd.read_csv(paths[-1]).macro_f1.iloc[0]
    return max(scores, key=scores.get)


def completed_comparison():
    """Summarize the latest completed runs in the three-stage protocol."""
    leader = leading_rgb_backbone()
    pairs = [(name, "original_rgb") for name in ("resnet18_baseline", *BACKBONES)]
    pairs.extend((leader, variant) for variant in ("lab_clahe", "bilateral_clahe"))
    records = []
    for name, variant in pairs:
        paths = sorted(
            (ROOT / "models" / "runs" / name / variant).glob("*/evaluation/test/metrics.csv")
        )
        if not paths:
            raise FileNotFoundError(f"Complete the run for {name}/{variant} first.")
        run = paths[-1].parents[2]
        row = pd.read_csv(run / "evaluation/validation/metrics.csv").iloc[0].to_dict()
        records.append(dict(model=name, input_variant=variant, run=run.name, **row))
    board = pd.DataFrame(records).sort_values(
        ["macro_f1", "defects_called_normal", "wrong"], ascending=[False, True, True]
    )
    board.to_csv(ROOT / "models/runs/validation_leaderboard.csv", index=False)
    return board